In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import natsort
from natsort import natsorted, ns
import numpy as np
import seaborn as sns
import glob
import os
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import scipy
import subprocess

In [ ]:
def plumed_to_pandas(filename="./COLVAR"):
    """
    Load a PLUMED file and save it to a dataframe.

    Parameters
    ----------
    filename : string, optional
        PLUMED output file

    Returns
    -------
    df : DataFrame
        Collective variables dataframe
    """
    skip_rows = 1
    # Read header
    headers = pd.read_csv(filename, sep=" ", skipinitialspace=True, nrows=0)
    # Discard #! FIELDS
    headers = headers.columns[2:]
    # Load dataframe and use headers for columns names
    df = pd.read_csv(
        filename,
        sep=" ",
        skipinitialspace=True,
        header=None,
        skiprows=range(skip_rows),
        names=headers,
        comment="#",
    )

    return df

In [ ]:
def extract_fundeltaF(file_path):
    """
    Extract the fundeltaF value from a file.

    Args:
    - file_path (str): The path to the file.

    Returns:
    - float: The fundeltaF value.
    """
    # Initialize fundeltaF variable
    fundeltaF = None

    # Open the file and read its content
    with open(file_path, 'r') as file:
        # Iterate through lines in the file
        for line in file:
            # Check if the line contains the 'fundeltaF' field
            if line.startswith('#! SET fundeltaF'):
                # Extract the fundeltaF value
                fundeltaF = round(float(line.split()[-1]), 3)
            if line.startswith('#! SET block_size'):
                # Extract the fundeltaF value
                sample_size = int(line.split()[-1])
            if line.startswith('#! SET DeltaF error'):
                # Extract the fundeltaF value
                deltaf_error = round(float(line.split()[-1]), 3)
            
            
    print(fundeltaF, sample_size, deltaf_error)
    return fundeltaF, sample_size, deltaf_error

In [ ]:
def get_blocks(colvar_path):
    df = read_colvar_file(colvar_path)
    n = len(df.time) # number of points
    nbs = 0 # initialize number of blocks
    prev_bs = 0 # previous block size
    nbr = 3 # minimum number of blocks 
    resol = 10 # resolution for spacing
    spacing = pow(2.0, 1.0 / resol) # block spacing criteria
    min_bs = int(round(n/99)) # define min block size so that you never have more than 99 blocks in total
    final_blocks = []
    while nbr <= n:
        bs = n // int(nbr)
        if bs != prev_bs and bs > min_bs:
            nb = n // bs
            final_blocks.append(int(nb))
            nbs += 1
        nbr *= spacing
        prev_bs = bs
    return final_blocks

In [ ]:
def analyser(colvar_path, lig, path=""):
    
    
    # These are the values for BRD4, modidfy as needed
    commands = {'lig1': [1.40, 3.4, 3.0, 1.8],
     'lig2': [1.40, 3.4, 3.0, 1.8],
     'lig3': [1.2, 3.4, 3.0, 1.8],
     'lig4': [1.4, 3.4, 3.0, 1.8],
     'lig5': [1.35, 3.4, 3.0, 1.8],
     'lig6': [1.33, 3.4, 3.0, 2.0],
     'ffmin_lig6': [1.33, 3.4, 3.0, 1.8],
     'lig7': [1.54, 3.4, 3.0, 1.8],
     'lig8': [1.25, 3.4, 3.0, 1.8],
     'lig9': [1.3, 3.4, 3.0, 2.0],
     'lig10': [1.0, 3.4, 3.0, 1.8],
     'lig11': [1.2, 3.4, 3.0, 1.8]}

    errors = []
    block_sizes = []
    colvar_name = os.path.splitext(os.path.basename(colvar_path))[0]
    blocks = get_blocks(colvar_path)
    min_, max_, uat, bat = commands[lig]
    for block in blocks[:7]:

        command = (
            f"python funnel_FES_from_Reweighting.py "
            f"--sigma 0.02 "
            f"--bias opes.bias "
            f"--colvar {colvar_path} "
            f"--cv pp.proj "
            f"--bin 200 "
            f"--temp 300 "
            f"--min {min_} "
            f"--max {max_} "
            f"--rfunnel 0.25 "
            f"--uat {uat} "
            f"--bat {bat} "
            f"--blocks {block} "
            f"--outfile {path}/fes_files/{colvar_name}_fes_blocks{block}.dat"
        )
        !{command}


        fundeltaF, sample_size, deltaf_error = extract_fundeltaF(f"{path}/fes_files/{colvar_name}_fes_blocks{block}.dat")
        
        errors.append(round(deltaf_error/4.184,3))
        block_sizes.append(sample_size)
        
        # Print the result
        print("fundeltaF (Kj/mol):", fundeltaF)
        print("fundeltaF (Kcal/mol):", round(fundeltaF/4.184,3))



    return round(fundeltaF/4.184,3), errors, block_sizes

In [ ]:
def clean_colvar(df):
    """
    Takes a DataFrame and keeps only the last row for each repeated value
    in the 'time' column.
    
    Parameters:
    df (pd.DataFrame): The input DataFrame.
    
    Returns:
    pd.DataFrame: A DataFrame with only the last row for each repeated 'time' value.
    """
    # Drop duplicates, keeping only the last occurrence
    result_df = df.drop_duplicates(subset='time', keep='last')
    return result_df

In [ ]:
def read_colvar_file(colvar_file):
    """
    Reads a COLVAR file and returns a pandas DataFrame with appropriate column names.

    Parameters:
    colvar_file (str): The path to the COLVAR file.

    Returns:
    pd.DataFrame: DataFrame containing the data from the COLVAR file with proper column names.
    """
    column_names = []

    # Open the file and extract the column names from the header line
    with open(colvar_file, 'r') as file:
        for line in file:
            if line.startswith('#! FIELDS'):
                # Extract column names from the header line
                column_names = line.strip().split()[2:]  # Split the line and remove the first two elements ('#!' and 'FIELDS')
                break

    # Load the COLVAR file into a pandas DataFrame with the correct column names
    df = pd.read_csv(colvar_file, delim_whitespace=True, comment='#', header=None, names=column_names)
    df = clean_colvar(df)
    return df

In [ ]:
def generate_colvars(df, time_start, output_path='./'):
    """
    Filters the DataFrame to include rows where the 'time' column is greater than the specified start time 
    and less than or equal to the current iteration time in nanoseconds, and writes the filtered DataFrame to 
    a new COLVAR file.

    Parameters:
    df (pd.DataFrame): The input DataFrame.
    time_start (float): The start time threshold in nanoseconds.
    output_path (str): The directory path for output COLVAR files.
    """
    
    # Get the last time entry in the DataFrame
    time_end = int(df["time"].iloc[-1])
    
    # Generate time intervals to iterate over, making sure to include the last data points
    time_end = int(df["time"].iloc[-1])
    eps = 100000 #time window to see when to stop the simulation
    time_intervals = list(range(int(time_start)+eps, time_end, eps)) + [time_end]
    
    for i in time_intervals:
        # Create the output filename
        output_filename = f"{output_path}{int(i / 1000)}_COLVAR.dat"
        print(output_filename)
        # Filter the DataFrame
        df_filtered = df[(df['time'] > time_start) & (df['time'] <= i)]

        # Prepare the FIELDS header line
        fields_header = "#! FIELDS " + " ".join(df.columns)

        # Write the filtered DataFrame to a new COLVAR file
        with open(output_filename, 'w') as file:
            file.write(fields_header + '\n')
            df_filtered.to_csv(file, sep=' ', index=False, header=False)

In [ ]:
def error_scan(colvars, lig, path=''):
    
    

    for colvar in colvars:
        results = []
        
        fundeltaF, errors, block_sizes = analyser(colvar_path=colvar, lig=lig,
                                                  path=path)
        results.append([fundeltaF, errors, block_sizes])
        colvar_name = os.path.splitext(os.path.basename(colvar))[0]
        print(colvar_name)
        np.save(f"{colvar_name}.npy", results)


In [ ]:
ligs = glob.glob("*lig*/") # The ligand dirs
ligs = [os.path.abspath(folder) for folder in ligs]
ligs = natsorted(ligs)
ligs

In [ ]:
for lig in ligs:
    # These are for BRD4, modify as needed
    times = {'lig1': 200000,
     'lig2': 400000,
     'lig3': 200000,
     'lig4': 300000,
     'lig5': 200000,
     'lig6': 205000,
     'ffmin_lig6': 200000,
     'lig7': 200000,
     'lig8': 200000,
     'lig9': 200000,
     'lig10': 400000,
     'lig11': 200000}
    
    lig_name = os.path.basename(lig)
   
    
    os.chdir(lig)
        
    # Create fes_files directory if it doesn't exist
    os.makedirs('colvars', exist_ok=True)
    
    #%mkdir -p fes_files/
    colvar_file = './COLVAR.0'  # Replace with the actual path to your COLVAR file
    df = read_colvar_file(colvar_file)
    
    start = times[lig_name]
    
    
    generate_colvars(df, time_start=start, output_path='./colvars/')
    
    
    os.chdir(lig+"/colvars/")

    colvars = glob.glob("./*COLVAR*.dat")
    colvars = [os.path.abspath(file) for file in colvars]
    colvars = natsorted(colvars)
    
    os.makedirs('fes_files', exist_ok=True)
    #%cd ~/Desktop/paper_hsp90/brd4/lig6/fes_files/
    %ls
    
    error_scan(colvars, lig_name,
           path=f"{lig}/colvars/")
    
    print(lig_name)

In [ ]:

def error_fit(a):
    t = a[0][2][::-1][-7:] # using only blocks [9, 8, 7, 6, 5, 4, 3]
    f = a[0][1][::-1][-7:] # using only blocks [9, 8, 7, 6, 5, 4, 3]

    T = t[-1] * 3

    param_bounds = (
        [0, 0, 1e-6, 1e-6],  # Lower bounds for sigma, alpha, tau_1, tau_2
        [1, np.inf, np.inf, np.inf]  # Upper bounds for sigma, alpha, tau_1, tau_2
    )

    def model_function(t, sigma, alpha, tau_1, tau_2):
        term1 = (2 * sigma**2) / T * (alpha * tau_1 * (1 + tau_1 / t * (np.exp(-t / tau_1) - 1)))
        term2 = (1 - alpha) * tau_2 * (1 + tau_2 / t * (np.exp(-t / tau_2) - 1))
        return term1 + term2

    try:
        popt, pcov = curve_fit(model_function, t, f, bounds=param_bounds)
        sigma, alpha, tau_1, tau_2 = popt
        if alpha < 0:
            raise ValueError("alpha is less than 0")
        #print(f"Fitted parameters: sigma = {sigma}, alpha = {alpha}, tau_1 = {tau_1}, tau_2 = {tau_2}")

        t_fit = np.linspace(min(t), max(t), 100)
        f_fit = model_function(t_fit, *popt)
    except (RuntimeError, ValueError) as e:
        print(f"Fitting failed: {e}")
        sigma, alpha, tau_1, tau_2 = None, None, None, None
        t_fit = np.linspace(min(t), max(t), 100)
        f_fit = None

    plt.scatter(t, f, label='Data')

    if f_fit is not None:
        plt.plot(t_fit, f_fit, label='Fitted curve', color='red')

    plt.xlabel('t')
    plt.ylabel('f(t)')
    plt.legend()
    plt.show()

In [ ]:
ligs = glob.glob("*lig*/") # folders of the ligands you are working on
ligs = [os.path.abspath(folder) for folder in ligs]
ligs = natsorted(ligs)
ligs

In [ ]:
for lig in ligs:
    print(str.upper(os.path.basename(lig)))
    errors = []
    deltas = []
    sorted_colvs = natsorted(glob.glob(f'{lig}/colvars/*COLVAR*.npy'))
    fit = False
    k = 0
 
    for col in sorted_colvs:
        try:
            a = np.load(col, allow_pickle=True)

            error_fit(a)
            k +=1 
        except (RuntimeError, ValueError):
            k = 0
            #t = a[0][2][::-1]
            #f = a[0][1][::-1]
            #plt.scatter(t, f, label='Data')
            #plt.show()
            continue

        if k == 2:
            start = os.path.basename(sorted_colvs[0])
            time = os.path.basename(col)
            lig_name = str.upper(os.path.basename(lig))
            print(start, time, lig_name)
        #    print("\n")
        #    break


In [ ]:
# These are for BRD4
exps = {'lig1': [9.8],
 'lig2': [9.6],
 'lig3': [9.0],
 'lig4': [8.9],
 'lig5': [8.8],
 'lig6': [8.2],
 'ffmin_lig6': [8.2],
 'lig7': [7.8],
 'lig8': [7.4],
 'lig9': [7.3],
 'lig10': [6.3],
 'lig11': [5.6]}

for lig in ligs:
    errors = []
    deltas = []
    sorted_colvs = natsorted(glob.glob(f'{lig}/colvars/*COLVAR*.npy'))
    for z,i in enumerate(sorted_colvs):
        temp = np.load(i, allow_pickle=True)
        delta = temp[0][0]
        error = max(temp[0][1][:7])
        print((z+1)*100,round(error,1), round(delta,1))
        errors.append(error)
        deltas.append(delta)
    lig_name = os.path.basename(lig)    
    exp = exps[lig_name][0]
    plt.errorbar(range(len(deltas)), deltas, yerr=errors, fmt='-o')
    plt.axhline(exp, linestyle='--', color='r')
    plt.axhline(exp-1, linestyle='--', color='purple')
    plt.axhline(exp+1, linestyle='--', color='purple')
    plt.title(f'{lig_name}')
    plt.show()
